# Baseline LightGCN run (M2)

Thin driver: all real code lives in the repo (`src/longtail_lightgcn`). This notebook only gets the code onto Kaggle, builds the data, runs the baseline and shows the results.

**Before running:** in the notebook settings turn on **Internet** (needed for `git clone` and the ML-1M download) and set the accelerator to a **GPU**.

## 1. Get the code
Clone the public repo and install it. `--no-deps` keeps Kaggle's preinstalled CUDA build of torch untouched.

In [ ]:
import os

REPO_URL = "https://github.com/rithikkulkarni/longtail-lightgcn.git"
COMMIT = ""  # optional: paste a commit hash to pin the exact code; empty = latest main

%cd /kaggle/working
if not os.path.exists("longtail-lightgcn"):
    !git clone $REPO_URL
%cd /kaggle/working/longtail-lightgcn
!git pull -q
if COMMIT:
    !git checkout -q $COMMIT
!pip install -e . --no-deps -q
!git rev-parse HEAD

## 2. Check the GPU

In [ ]:
import torch
print("torch", torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 3. Build the data
Downloads ML-1M and writes `data/processed/split_{0..4}.npz` plus `items.csv`. Prints the split statistics (including how many items have zero train edges).

In [ ]:
!python scripts/prepare_data.py

## 4. Train the baseline
Defaults come from `TrainConfig` (the authors' README settings). Any field can be overridden here, e.g. `--lr 0.005 --max_epochs 300`. Training evaluates validation Recall@20 every 20 epochs and early-stops after 5 evaluations without improvement; test Recall@20 is computed once at the end.

In [ ]:
!python -m longtail_lightgcn.run_baseline --split_seed 0 --seed 0 --out_dir /kaggle/working/results/baseline_split0_seed0

## 5. Look at the results

In [ ]:
import json
import pandas as pd

res = json.load(open("/kaggle/working/results/baseline_split0_seed0/results.json"))
print("commit:", res["git_commit"])
print("best epoch:", res["best_epoch"])
print("val recall@20:", res["val_recall@20"])
print("test recall@20:", res["test_recall@20"])
pd.DataFrame(res["history"])